# Risk models: comparing classifiers

Four classifiers (logistic regression, random forest, gradient boosting, MLP)
are trained to predict the primary outcome (death or dependency at 6 months)
from the baseline features. The treatment under study, aspirin, is not among
the features.

The goal is to choose the base classifier for the T-learner. Models are
compared with 5-fold cross-validation on the training set, on AUC
(discrimination), log-loss and Brier score (calibration). The test set is
not used.

In [5]:
from pathlib import Path

import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline

from ml_med.modeling import make_preprocessor, prepare_data, split_train_test

FEATURES_PATH = Path("../data/processed/ist_features.parquet")

In [2]:
df = pd.read_parquet(FEATURES_PATH)
X, y, t = prepare_data(df, "RXASP")
X_train, X_test, y_train, y_test, t_train, t_test = split_train_test(X, y, t)

In [6]:
def make_model(clf):
    return Pipeline([("prep", make_preprocessor("RXASP")), ("clf", clf)])


candidates = {
    "logistic": LogisticRegression(max_iter=1000),
    "random_forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=20, random_state=0, n_jobs=-1
    ),
    "gradient_boosting": HistGradientBoostingClassifier(random_state=0),
    "MLP": MLPClassifier(
        hidden_layer_sizes=(64, 32),
        alpha=1e-3,
        early_stopping=True,
        max_iter=500,
        random_state=0,
    ),
}
scoring = ["roc_auc", "neg_log_loss", "neg_brier_score"]

In [7]:
rows = []
for name, clf in candidates.items():
    cv_results = cross_validate(
        make_model(clf), X_train, y_train, cv=5, scoring=scoring
    )
    rows.append(
        {
            "model": name,
            "auc": cv_results["test_roc_auc"].mean(),
            "log_loss": -cv_results["test_neg_log_loss"].mean(),
            "brier": -cv_results["test_neg_brier_score"].mean(),
        }
    )

results = pd.DataFrame(rows).set_index("model").round(4)
results

,auc,log_loss,brier
model,,,
logistic,0.7994,0.5205,0.1742
random_forest,0.7980,0.5246,0.1755
gradient_boosting,0.7980,0.5223,0.1747
MLP,0.7953,0.5266,0.1762


Conclusion: the models give similar results, and logistic regression is a good choice. 